# 02 – Variable categorization

Each variable is classified as identifier, independent (input) or dependent (outcome), and as controllable or non-controllable by the operator. The classification determines which columns are used as model inputs and which failure causes can be prevented by changing machine settings.

In [1]:
import pandas as pd

from config import CLEAN_DATA, RESULTS_DIR

pd.set_option("display.max_colwidth", None)

df = pd.read_csv(CLEAN_DATA)
sensors = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]

## 1. Relationships between the input variables

In [2]:
df[sensors].corr().round(3)

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
Air temperature [K],1.000,0.876,0.023,-0.015,0.014
Process temperature [K],0.876,1.000,0.020,-0.015,0.014
Rotational speed [rpm],0.023,0.020,1.000,-0.875,-0.000
Torque [Nm],-0.015,-0.015,-0.875,1.000,-0.003
Tool wear [min],0.014,0.014,-0.000,-0.003,1.000


In [3]:
temp_diff = df["Process temperature [K]"] - df["Air temperature [K]"]
temp_diff.describe().round(2)

count    9973.0
mean       10.0
std         1.0
min         7.6
25%         9.3
50%         9.8
75%        11.0
max        12.1
dtype: float64

Process temperature follows air temperature (r ≈ 0.88) and stays about 10 K above it, so it is not an independent machine setting. Rotational speed and torque are strongly negatively correlated (r ≈ −0.88), so they behave as one coupled operating point. Tool wear is uncorrelated with all other variables.

## 2. Categorization table

In [4]:
variables = pd.DataFrame(
    [
        ("UDI", "Identifier", "-", "No", "Row index without physical meaning."),
        ("Product ID", "Identifier", "-", "No", "Serial number; the quality variant it encodes is already given by Type."),
        ("Type", "Independent", "Controllable", "Yes", "Product variant (L/M/H) is chosen in production planning."),
        ("Air temperature [K]", "Independent", "Non-controllable", "Yes", "Ambient condition of the workshop."),
        ("Process temperature [K]", "Independent", "Non-controllable", "Yes", "Follows air temperature at about +10 K; not a machine setting."),
        ("Rotational speed [rpm]", "Independent", "Controllable", "Yes", "Machine set point."),
        ("Torque [Nm]", "Independent", "Controllable", "Yes", "Load set by the process parameters; coupled with rotational speed."),
        ("Tool wear [min]", "Independent", "Indirectly controllable", "Yes", "Accumulates with use; reset by the tool replacement policy."),
        ("Machine failure", "Dependent", "-", "Target", "Outcome to predict."),
        ("TWF, HDF, PWF, OSF, RNF", "Dependent", "-", "No", "Failure modes that define the target; used only to analyse results, as inputs they would leak the target."),
    ],
    columns=["Variable", "Role", "Controllability", "Model input", "Justification"],
)
variables.to_csv(RESULTS_DIR / "02_variable_categories.csv", index=False)
variables

,Variable,Role,Controllability,Model input,Justification
0,UDI,Identifier,-,No,Row index without physical meaning.
1,Product ID,Identifier,-,No,Serial number; the quality variant it encodes is already given by Type.
2,Type,Independent,Controllable,Yes,Product variant (L/M/H) is chosen in production planning.
3,Air temperature [K],Independent,Non-controllable,Yes,Ambient condition of the workshop.
4,Process temperature [K],Independent,Non-controllable,Yes,Follows air temperature at about +10 K; not a machine setting.
5,Rotational speed [rpm],Independent,Controllable,Yes,Machine set point.
6,Torque [Nm],Independent,Controllable,Yes,Load set by the process parameters; coupled with rotational speed.
7,Tool wear [min],Independent,Indirectly controllable,Yes,Accumulates with use; reset by the tool replacement policy.
8,Machine failure,Dependent,-,Target,Outcome to predict.
9,"TWF, HDF, PWF, OSF, RNF",Dependent,-,No,"Failure modes that define the target; used only to analyse results, as inputs they would leak the target."
